# Where should I look for stellar oscillations?

In this notebook we use AsteroScale to predict some typical quantities about the oscillations in a solar-like oscillator. These include the characteristic frequency of the oscillations $\nu_{\mathrm{max}}$, the frequency band they appear in around $\nu_{\mathrm{max}}$ and their amplitude as it would appear in a time series. Note oscillations are visible in both radial velocity and photometric time series, however in AsteroScale we currently only consider photometric variability.

In the following we'll just use AsteroScale to make point estimates. You can of course supply measurement uncertainties to obtain properly marginalized distributions.

In this case we let's pretend we know what kind of star we're looking at. We can then calculate the characteristics of he frequencies pretty easily using the asteroseismic scaling relations.

In [6]:
import asteroscale as ast

star = {"M": 1.1, "R": 1.3, "Teff": 6000, "FeH": 0.1}

prediction = ast.solve(
    star,
    want=["numax", "dnu", "FWHM_env", "A_env",],
    bandpass="TESS",
)
prediction

{'numax': np.float64(1982.2491458473373),
 'dnu': np.float64(94.85782037346993),
 'FWHM_env': np.float64(598.0358941076695),
 'A_env': np.float64(2.9642737456372936)}

The oscillation power is approximately distributed in frequency according to a Gaussian centered on `numax`. If you have a power spectrum of time series from your star, as observed for example by TESS, to see if the oscillations are particularly strong a useful first search interval is therefore approximately `numax ± FWHM_env/2`, although this is not a hard boundary. `A_env` is the maximum radial-mode RMS amplitude, not the total light-curve RMS.

In [2]:
lower = prediction["numax"] - prediction["FWHM_env"] / 2
upper = prediction["numax"] + prediction["FWHM_env"] / 2
print(f"Suggested search interval: {lower:.0f}–{upper:.0f} microhertz")

Suggested search interval: 1683–2281 microhertz


## TESS and Kepler amplitudes

The amplitudes of the oscillations decreases when observed in redder filters, so oscillations observed by Kepler are slightly larger than TESS because its spectral response function is bluer. 

In [4]:
for bandpass in ("TESS", "Kepler"):
    amplitude = ast.solve(star, want=["A_env"], bandpass=bandpass)["A_env"]
    print(f"{bandpass:6s}: {amplitude:.2f} ppm")

TESS  : 2.96 ppm
Kepler: 3.53 ppm


## Hot stars have lower amplitudes 

For stars a few hundred Kelvin hotter than the Sun we see the oscillations decrease in amplitude as we approach what is known as the classical instability strip. This roughly delineates the cooler solar-like oscillators and the hotter $\gamma$-dor and $\delta$-scuti type pulsators. 

**Note** AsteroScale will not provide you with any sensible predictions for these hotter types of stars since they don't follow the same physics.

In [9]:
star = {"M": 1.1, "R": 1.3, "Teff": 7200, "FeH": 0.1}

prediction = ast.solve(
    star,
    want=["numax", "dnu", "FWHM_env", "A_env"],
    bandpass="TESS",
)
prediction

{'numax': np.float64(1809.537619626558),
 'dnu': np.float64(90.63755013598447),
 'FWHM_env': np.float64(901.5058834654263),
 'A_env': np.float64(1.9495766102104717)}

Note that predicted signal is not necessarily detectable or important for the variance you see in the light curve. Compare the oscillatoin frequency range with the Nyquist frequency of your observations and resolution of the observations, and the amplitude of the oscillations with the instrumental noise.